#  `results_pathological/` per algorithm

Pathological is not a separate dataset — it's just the 4 `P001_1`–`P004_1`
subjects inside `myosegmenTUM/`. Each algorithm's `column_compare_*.ipynb`
notebook already writes per-muscle raw CSVs (one row per subject/stack) into
a `results_water` / `results_fat_frac` / `results_dixon` / `results_channels`
folder. This notebook filters those CSVs down to only the `P*` subjects and
writes them into a new `results_pathological/` folder next to the source,
using the same filenames — so the same `per_subject_dice()`-style code we
already use for AIPS/Sheffield/Augmented can point at `results_pathological`
unchanged.

Every modality folder that exists per algorithm is processed (not just one),
 the filtered
output keeps the original filename (which already encodes the modality), so
nothing is silently collapsed or overwritten.

**Caveats:** 
- `medsegdiff` results are now disincluded

In [1]:
import os
import re
import glob
import pandas as pd

In [2]:
# ── Registry: (algorithm label, base dir, [modality subfolders to check]) ─────────
# base dir is relative to this notebook's location (eval_notebooks/).
# Only MyoSegmenTUM/Pathological modality folders are listed — Sheffield/AIPS/
# Augmented folders (results_sheffield, results_asian_water, results_augmented,
# augmented_results, ...) are intentionally excluded.
REGISTRY = [
    ('MuscleMap WB',           'muscle_map_wb/codes',        ['results_water', 'results_fatfrac']),
    ('MuscleMap Thigh',        'muscle_map_thigh/codes',     ['results_water', 'results_fat_frac']),
    ('Dafne',                  'dafne',                      ['results_water', 'results_fat_frac']),
    ('MuSeg',                  'museg',                      ['results_water', 'results_fat_frac', 'results_dixon']),
    ('Multimodal-Multiethnic', 'multimodal-multiethnic/results', ['results_water', 'results_fatfrac']),
    ('MedCLIP-SAMv2',          'medclipsamv2textboxes',      ['results_water', 'results_fat_frac']),
]

# Subject/stack are embedded either as their own columns (dafne/museg/
# medclipsamv2textboxes style) or inside a pred_label/image filename
# (musclemap_wb/musclemap_thigh/hirriririir style), e.g.
# 'HV003_2_WATER_stack1_dseg.nii.gz' or '..\\myosegmenTUM\\P001_1\\...\\combined_gt_stack1.mha'.
SUBJECT_RE = re.compile(r'([A-Za-z]+\d+_\d+)_(?:WATER|FATFRACTION)_stack\d+')
SUBJECT_RE_PATH = re.compile(r'[\\/]([A-Za-z]+\d+_\d+)[\\/]SegmentationMasks')


def extract_subject(row):
    """Return the subject id (e.g. 'P002_1') for one CSV row, or None."""
    if 'subject' in row and pd.notna(row['subject']):
        return str(row['subject'])
    for col in ('pred_label', 'pred_file', 'image', 'gt_path'):
        if col in row and pd.notna(row[col]):
            m = SUBJECT_RE.search(str(row[col])) or SUBJECT_RE_PATH.search(str(row[col]))
            if m:
                return m.group(1)
    return None


def is_pathological(subject):
    return subject is not None and subject.upper().startswith('P')

In [5]:
stats = []  # rows for the final summary table

for algo_label, base_dir, modalities in REGISTRY:
    for modality in modalities:
        src_dir = os.path.join(base_dir, modality)
        if not os.path.isdir(src_dir):
            stats.append(dict(algorithm=algo_label, modality=modality, status='folder missing',
                               csv=None, n_total=0, n_pathological=0))
            continue

        csv_files = sorted(glob.glob(os.path.join(src_dir, 'df_*.csv')))
        if not csv_files:
            stats.append(dict(algorithm=algo_label, modality=modality, status='no CSVs',
                               csv=None, n_total=0, n_pathological=0))
            continue

        # Modality-specific subfolder — some algorithms (e.g. hirriririir) reuse the
        # exact same per-muscle filenames across modalities, which would silently
        # clobber each other if written into one flat results_pathological/ folder.
        dest_dir = os.path.join(base_dir, 'results_pathological', modality)
        os.makedirs(dest_dir, exist_ok=True)

        for csv_path in csv_files:
            df = pd.read_csv(csv_path, index_col=0)
            n_total = len(df)

            if n_total == 0:
                stats.append(dict(algorithm=algo_label, modality=modality, status='source empty',
                                   csv=os.path.basename(csv_path), n_total=0, n_pathological=0))
                continue

            subjects = df.apply(extract_subject, axis=1)
            mask = subjects.apply(is_pathological)
            filtered = df[mask]

            out_path = os.path.join(dest_dir, os.path.basename(csv_path))
            filtered.to_csv(out_path)

            status = 'ok' if len(filtered) > 0 else 'zero pathological rows matched'
            stats.append(dict(algorithm=algo_label, modality=modality, status=status,
                               csv=os.path.basename(csv_path),
                               n_total=n_total, n_pathological=len(filtered)))

print(f'Processed {len(stats)} (algorithm, modality, csv) combinations.')

Processed 52 (algorithm, modality, csv) combinations.


In [6]:
# ── Summary ───────────────────────────────────────────────────────────────────
summary = pd.DataFrame(stats)
display(summary)

problems = summary[summary['status'] != 'ok']
if len(problems):
    print(f'\n{len(problems)} row(s) need attention:')
    display(problems)
else:
    print('\nEverything processed cleanly.')

,algorithm,modality,status,csv,n_total,n_pathological
0,MuscleMap WB,results_water,ok,df_L_gracilis_musclemap_wb_water.csv,46,12
1,MuscleMap WB,results_water,ok,df_L_sartorius_musclemap_wb_water.csv,46,12
2,MuscleMap WB,results_water,ok,df_R_gracilis_musclemap_wb_water.csv,46,12
3,MuscleMap WB,results_water,ok,df_R_sartorius_musclemap_wb_water.csv,46,12
4,MuscleMap WB,results_fatfrac,ok,df_L_gracilis_musclemap_wb_fatfrac.csv,54,12
5,MuscleMap WB,results_fatfrac,ok,df_L_sartorius_musclemap_wb_fatfrac.csv,54,12
6,MuscleMap WB,results_fatfrac,ok,df_R_gracilis_musclemap_wb_fatfrac.csv,54,12
7,MuscleMap WB,results_fatfrac,ok,df_R_sartorius_musclemap_wb_fatfrac.csv,54,12
8,MuscleMap Thigh,results_water,ok,df_L_gracilis_musclemap_thigh_water.csv,46,12
9,MuscleMap Thigh,results_water,ok,df_L_sartorius_musclemap_thigh_water.csv,46,12



Everything processed cleanly.
